# Exercise 1: Simple Neural Network on HR Dataset

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)
tf.random.set_seed(42)

## 2. Load the Dataset

In [ ]:
import urllib.request

# The HR dataset is publicly available via this mirror
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/HR_comma_sep.csv"
urllib.request.urlretrieve(url, "hr_comma_sep.csv")

df = pd.read_csv("hr_comma_sep.csv")

print("Shape:", df.shape)
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df["left"].value_counts())

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df["left"].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue", "tomato"])
axes[0].set_title("Employee Left (Target)")
axes[0].set_xticklabels(["Stayed", "Left"], rotation=0)

df.groupby("left")["satisfaction_level"].mean().plot(
    kind="bar", ax=axes[1], color=["steelblue", "tomato"])
axes[1].set_title("Avg Satisfaction by Left")
axes[1].set_xticklabels(["Stayed", "Left"], rotation=0)

df.groupby("left")["average_montly_hours"].mean().plot(
    kind="bar", ax=axes[2], color=["steelblue", "tomato"])
axes[2].set_title("Avg Monthly Hours by Left")
axes[2].set_xticklabels(["Stayed", "Left"], rotation=0)

plt.tight_layout()
plt.show()

## 3. Encode Labels and Scale Features

In [ ]:
data = df.copy()

# Encode categorical columns with LabelEncoder
le = LabelEncoder()

if "sales" in data.columns:
    data["sales"] = le.fit_transform(data["sales"])

if "Department" in data.columns:
    data["Department"] = le.fit_transform(data["Department"])

if "salary" in data.columns:
    data["salary"] = le.fit_transform(data["salary"])

print("After encoding:")
print(data.dtypes)
data.head()

In [ ]:
X = data.drop(columns=["left"])
y = data["left"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Target shape: {y.shape}")

## 4. Split the Dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")

## 5. Create the Neural Network Model with Sequential()

In [ ]:
n_features = X_train.shape[1]

model = Sequential([
    # Input layer
    Dense(64, activation="relu", input_shape=(n_features,)),
    Dropout(0.3),

    # Hidden layer 1
    Dense(128, activation="relu"),
    Dropout(0.3),

    # Hidden layer 2
    Dense(64, activation="relu"),
    Dropout(0.2),

    # Output layer — sigmoid for binary classification
    Dense(1, activation="sigmoid")
])

model.summary()

## 6. Compile and Train the Model

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Val Loss")
axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train Accuracy")
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy")
axes[1].set_title("Accuracy over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Evaluate the Model

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
import seaborn as sns

y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Stayed", "Left"]))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d",
            cmap="Blues", cbar=False,
            xticklabels=["Stayed", "Left"],
            yticklabels=["Stayed", "Left"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()